In [1]:
import json
from typing import TypedDict
import os
from dotenv import load_dotenv

# Load variables from .env (if present) and fallback to existing environment variables
load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY is not set. Add it to your environment or .env file.")

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY is not set. Add it to your environment or .env file.")

In [ ]:
import dspy

# Connect to and set an LLM
lm = dspy.LM('anthropic/claude-haiku-4-5', api_key=ANTHROPIC_API_KEY, max_tokens=64000)
dspy.configure(lm=lm)

# Test
lm("Say hello")

['Hello! 👋 How can I help you today?']

In [7]:
from tavily import TavilyClient
from typing import Literal

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

def internet_search(query: str, max_results: int = 5, include_raw_content: bool = False): 
	"""Run a web search""" 
	return tavily_client.search(
		query, max_results=max_results, 
		include_raw_content=include_raw_content, 
		topic="general",
	)

def read_webpage(url: str) -> dict:
	"""Read the content of a URL."""
	response = tavily_client.extract(url)
	return response

In [10]:
# Set our budget
BUDGETS = {
    "light":  {"max_subtopics": 3, "num_sources": 2, "max_search_results": 3},
    "medium": {"max_subtopics": 5, "num_sources": 2, "max_search_results": 5},
    "deep":   {"max_subtopics": 7, "num_sources": 5, "max_search_results": 10},
}
budget = BUDGETS["light"]

# Output destinations
ARTIFACT_OUTPUT = "output/research_artifact.json"
REPORT_OUTPUT = "output/workflow.md"

In [11]:
research_request = "Write a short explanation about Recrusive Language Models (RLMs)"


In [12]:
# Create our clarifier program
class ClarifyingSignature(dspy.Signature):
    "Generate clarifying questions to inform research work in response to the request."
    research_request: str = dspy.InputField()
    number_of_questions: int = dspy.InputField(desc="The number of clarifying questions to generate")
    clarifying_questions: list[str] = dspy.OutputField()

clarifier = dspy.Predict(ClarifyingSignature)

In [13]:
# Generate our questions
results = clarifier(research_request=research_request, number_of_questions=3)

# Get our answers
q_and_a = []
for question in results.clarifying_questions:
    answer = input(f"{question}\n")
    q_and_a.append({"clarifying_question": question, "user_guidance": answer})

In [14]:
q_and_a

[{'clarifying_question': "What is your target audience's level of expertise with language models and machine learning concepts?",
  'user_guidance': 'junior and senior engineers'},
 {'clarifying_question': 'Should the explanation focus on the theoretical foundations of RLMs, their practical applications, or both?',
  'user_guidance': 'both'},
 {'clarifying_question': 'Are there specific use cases or domains (e.g., code generation, mathematical reasoning, creative writing) you want the explanation to emphasize?',
  'user_guidance': 'accurate data retrieval in large context docs'}]